# Data Enc: Data Preprocessing – Skewness, Scaling and Encoding

## Aim
Study skewness, apply Min-Max scaling and standardization, and encode ordinal and nominal categorical features using the urban-climate dataset.

## Dataset
`urban_climate.csv` contains 11,040 records and 20 attributes. The dataset includes numerical and categorical features.

## 1. Load and inspect the dataset
Load the CSV file and inspect its first rows, shape, data types and missing values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df = pd.read_csv('urban_climate.csv')

print(df.head())
print('Shape:', df.shape)
print(df.dtypes)
print(df.isnull().sum())

## 2. Skewness, histogram and KDE
Skewness indicates asymmetry in a numerical distribution. A positive value indicates a longer right tail; a negative value indicates a longer left tail. Histograms and KDE curves help visualize the distributions.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
skewness = df[numeric_cols].skew()
print(skewness)

for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title('Histogram and KDE - ' + col)
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## 3. Min-Max scaling manually and with Scikit-Learn
Formula: `X_scaled = (X - X_min) / (X_max - X_min)`. The manual Pandas implementation is compared with `MinMaxScaler`.

In [ ]:
scale_cols = ['temperature_celsius', 'humidity_percent', 'precipitation_mm', 'wind_speed_ms', 'urban_heat_island_intensity']
scale_cols = [c for c in scale_cols if c in df.columns]

manual_scaled = df[scale_cols].copy()
for col in scale_cols:
    minimum, maximum = df[col].min(), df[col].max()
    manual_scaled[col] = (df[col] - minimum) / (maximum - minimum) if maximum != minimum else 0.0

minmax_scaler = MinMaxScaler()
sklearn_scaled = pd.DataFrame(minmax_scaler.fit_transform(df[scale_cols]), columns=scale_cols, index=df.index)
print('Manual scaling:')
print(manual_scaled.head())
print('Scikit-Learn scaling:')
print(sklearn_scaled.head())
print('Minimum values:')
print(sklearn_scaled.min())
print('Maximum values:')
print(sklearn_scaled.max())
print('Maximum difference between manual and sklearn:', (manual_scaled - sklearn_scaled).abs().to_numpy().max())

## 4. Ordinal encoding
The source dataset does not contain a pre-existing ordinal feature such as education level. A derived precipitation category is created from precipitation quantiles solely to demonstrate ordinal encoding: Low = 0, Moderate = 1, High = 2.

In [ ]:
df_encoded = df.copy()
df_encoded['precipitation_level'] = pd.qcut(
    df_encoded['precipitation_mm'].rank(method='first'),
    q=3,
    labels=['Low', 'Moderate', 'High']
)
ordinal_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}
df_encoded['precipitation_level_encoded'] = df_encoded['precipitation_level'].map(ordinal_mapping).astype(int)
print(df_encoded[['precipitation_mm', 'precipitation_level', 'precipitation_level_encoded']].head())

## 5. One-Hot encoding
Nominal categories have no natural order. `get_dummies()` converts them into indicator columns. `drop_first=True` removes one reference category per feature to avoid redundant dummy columns (the Dummy Variable Trap in a full-rank linear model with an intercept).

In [ ]:
nominal_cols = [c for c in ['city', 'country', 'continent', 'subregion', 'data_source', 'capital_city', 'iso3_code', 'currency'] if c in df_encoded.columns]
encoded_df = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True, dtype=int)
print('Nominal columns encoded:', nominal_cols)
print(encoded_df.head())

## 6. Standardization and comparison
Standardization uses `z = (x - mean) / population_std`. `StandardScaler` produces approximately mean 0 and population standard deviation 1. The first five rows and shapes of the original and transformed datasets are displayed.

In [ ]:
standardized_df = encoded_df.copy()
standard_scaler = StandardScaler()
standardized_df[numeric_cols] = standard_scaler.fit_transform(standardized_df[numeric_cols])

print('Standardized means:')
print(standardized_df[numeric_cols].mean().round(4))
print('Standardized population standard deviations:')
print(standardized_df[numeric_cols].std(ddof=0).round(4))

print('Original dataset:')
print(df.head())
print('Transformed dataset:')
print(standardized_df.head())
print('Original shape:', df.shape)
print('Transformed shape:', standardized_df.shape)

normalized_df = df.copy()
normalized_df[scale_cols] = sklearn_scaled
normalized_df.to_csv('urban_climate_minmax_scaled.csv', index=False)
standardized_df.to_csv('urban_climate_preprocessed.csv', index=False)
print('Saved transformed CSV files.')

## Conclusion
The experiment inspected numerical distributions using skewness, histograms and KDE. Min-Max scaling was implemented manually and with Scikit-Learn. A derived ordinal precipitation feature demonstrated ordered label mapping, while nominal features were one-hot encoded with `drop_first=True`. StandardScaler was used to verify mean approximately 0 and population standard deviation approximately 1. Min-Max scaled features were verified to have minimum 0 and maximum 1.